# 02 · Préparer les données et construire un protocole d'évaluation

**Étapes 3–4** du cas pratique · Slides 18–19, 22–23, 26–28, 34–37.

**Règle de ce notebook.** Le jeu de test est mis sous scellé au début de l'étape 4. D'ici là, **aucune transformation n'est ajustée sur les données réelles** : on n'applique aux vraies lignes que des transformations **sans apprentissage** (`AddFeatures`), et on illustre les transformations qui apprennent sur des **données synthétiques**.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.linear_model import LogisticRegression

from financial_inclusion_africa import config, features
from financial_inclusion_africa.data import CAT_COLS, DROPPED_COLS, NUM_COLS, clean, load_raw
from financial_inclusion_africa.features import AddFeatures, feature_columns, onehot_preprocessor, pipe

set_config(display="diagram")
pd.set_option("display.max_columns", 30)

## Étape 3 · Préparer les données et construire les variables

### 3.1 Séparer X et y, retirer les colonnes interdites

`clean()` (dans `data.py`) sépare la cible et retire les colonnes qui ne doivent pas servir de variables explicatives, pour les raisons établies à l'étape 2.

In [2]:
df = load_raw()
X, y = clean(df)

print(f"X : {X.shape[0]:,} lignes × {X.shape[1]} colonnes · y : {y.mean():.2%} de positifs")
display(pd.Series(DROPPED_COLS, name="raison").rename_axis("colonne retirée").to_frame())
pd.DataFrame({"type": ["catégorielle"] * len(CAT_COLS) + ["numérique"] * len(NUM_COLS)},
             index=CAT_COLS + NUM_COLS).T

X : 23,524 lignes × 10 colonnes · y : 14.08% de positifs


,raison
colonne retirée,
uniqueid,"identifiant arbitraire, numérotation par pays,..."
year,"entièrement déterminée par country, valeur fut..."


,country,location_type,cellphone_access,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type,household_size,age_of_respondent
type,catégorielle,catégorielle,catégorielle,catégorielle,catégorielle,catégorielle,catégorielle,catégorielle,numérique,numérique


In [3]:
assert X.shape == (23_524, 10)
assert {"uniqueid", "year", config.TARGET}.isdisjoint(X.columns)
assert y.sum() == 3_312

### 3.2 Feature engineering

`AddFeatures` (dans `features.py`) ajoute des variables qui portent une **idée métier**. Ce sont des règles fixes, écrites à la main : elles **n'apprennent rien des données**, on peut donc les appliquer à n'importe quelle ligne sans risque de fuite.

| Variable | Définition | Idée métier · constat de l'étape 2 |
|---|---|---|
| `education_rank` | 0 (aucune) → 4 (supérieur) ; **NaN** pour *Other/Dont know/RTA* | L'éducation agit dans un ordre (H1) ; la modalité inconnue n'est ni la plus haute ni la plus basse (Q2.6) |
| `education_unknown` *(ajoutée, Q2.6)* | 1 si *Other/Dont know/RTA* | Garde l'information « ne sait pas » après imputation du rang |
| `is_head` | 1 si chef de ménage | 17,7 % ont un compte, contre 6,0 % des parents du chef |
| `formal_job` | 1 si salarié du privé ou de l'État | Salaire versé sur un compte (H2) |
| `urban_phone` | 1 si en ville **et** avec téléphone | Accès physique et numérique aux services |
| `age_group` | 16-25, 26-35, 36-50, 51-65, 66+ | Relation en U inversé ; âges arrondis |
| **`phone_education`** *(la nôtre)* | `education_rank` si téléphone, sinon 0 | Interaction téléphone × éducation (H3) |
| **`lives_alone`** *(la nôtre)* | 1 si ménage d'une personne | 19,1 % ont un compte, contre 14,1 % |

In [4]:
# Sans apprentissage : on peut l'appliquer directement aux vraies lignes
Xf = AddFeatures().fit_transform(X)
new_cols = [c for c in Xf.columns if c not in X.columns]
# quelques lignes, dont une personne d'éducation inconnue
rows = [0, 1, 2, X.index[X["education_level"].eq("Other/Dont know/RTA")][0]]
Xf.loc[rows, ["education_level", "cellphone_access", "household_size", *new_cols]]

,education_level,cellphone_access,household_size,education_rank,education_unknown,is_head,formal_job,urban_phone,age_group,phone_education,lives_alone
0,Secondary education,Yes,3,2.0,0,0,0,0,16-25,2.0,0
1,No formal education,No,5,0.0,0,1,0,0,66+,0.0,0
2,Vocational/Specialised training,Yes,5,3.0,0,0,0,1,26-35,3.0,0
58,Other/Dont know/RTA,Yes,11,NaN,1,0,0,0,16-25,NaN,0


In [5]:
# Contrôle : valeurs prises par les nouvelles variables (effectifs, aucune cible utilisée)
binaires = ["education_unknown", "is_head", "formal_job", "urban_phone", "lives_alone"]
display(Xf[binaires].agg(["sum", "mean"]).T.set_axis(["nombre de 1", "part"], axis=1).round(3))
display(pd.concat({
    "education_rank": Xf["education_rank"].value_counts(dropna=False).sort_index(),
    "phone_education": Xf["phone_education"].value_counts(dropna=False).sort_index(),
}, axis=1).fillna(0).astype(int).rename_axis("valeur"))
Xf["age_group"].value_counts().sort_index().to_frame("effectif")

,nombre de 1,part
education_unknown,35.0,0.001
is_head,12831.0,0.545
formal_job,1442.0,0.061
urban_phone,6384.0,0.271
lives_alone,3057.0,0.130


,education_rank,phone_education
valeur,,
0.0,4515,8578
1.0,12791,9441
2.0,4223,3608
3.0,803,781
4.0,1157,1081
NaN,35,35


,effectif
age_group,
16-25,5601
26-35,6573
36-50,6055
51-65,3329
66+,1966


### 3.3 Assembler un pipeline

Le pipeline enchaîne trois blocs. Les deux derniers **apprennent** ; ils seront ajustés sur les seules lignes d'entraînement de chaque pli.

1. `AddFeatures` : variables métier, sans apprentissage ;
2. `ColumnTransformer` : one-hot des catégorielles (`min_frequency=20`, `handle_unknown="infrequent_if_exist"`), imputation médiane puis `StandardScaler` des numériques ;
3. le modèle : ici notre **modèle de base**, la régression logistique.

`pipe(model, features=...)` construit trois variantes, comparées à l'étape 5 : `"none"` (colonnes brutes), `"guide"` (variables du guide), `"all"` (guide + nos variables).

In [6]:
logreg = pipe(LogisticRegression(max_iter=2000))
logreg

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('addfeatures', ...), ('columntransformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,custom,True
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_thresh

In [7]:
pd.DataFrame(
    {f: [len(feature_columns(f)[0]), len(feature_columns(f)[1])] for f in ["none", "guide", "all"]},
    index=["colonnes catégorielles", "colonnes numériques"],
)

,none,guide,all
colonnes catégorielles,8,9,9
colonnes numériques,2,7,9


> **Piège · CatBoost et scikit-learn** (étape 6). CatBoost traite les catégories nativement : on lui passe les colonnes texte telles quelles, sans one-hot. Avec scikit-learn 1.9, `CatBoostClassifier(cat_features=[...])` avec une **liste** ne peut pas être cloné et `cross_validate` échoue ; il faut un **tuple** : `CatBoostClassifier(cat_features=tuple(CAT_COLS), verbose=0)`.

### Q3.1 · Quelles transformations apprennent quelque chose des données ?

In [8]:
# Données synthétiques au schéma du vrai jeu : on ne touche pas aux vraies lignes
rng = np.random.default_rng(config.RANDOM_STATE)

def make_toy(n, age_mean):
    sample = lambda col: rng.choice(X[col].unique(), n)  # seul le schéma est utilisé
    toy = pd.DataFrame({c: sample(c) for c in CAT_COLS})
    toy["household_size"] = rng.integers(1, 10, n)
    toy["age_of_respondent"] = rng.normal(age_mean, 12, n).clip(16, 100).round().astype(int)
    return toy

cat, num = feature_columns("all")
learned = {}
for name, toy in {"pli A (âge moyen 30)": make_toy(400, 30), "pli B (âge moyen 50)": make_toy(400, 50)}.items():
    pre = onehot_preprocessor(cat, num).fit(AddFeatures().fit_transform(toy))
    scaler = pre.named_transformers_["num"][-1]
    imputer = pre.named_transformers_["num"][0]
    learned[name] = {
        "moyenne apprise de l'âge (StandardScaler)": scaler.mean_[num.index("age_of_respondent")],
        "écart-type appris de l'âge": scaler.scale_[num.index("age_of_respondent")],
        "médiane apprise de education_rank (SimpleImputer)": imputer.statistics_[num.index("education_rank")],
        "modalités apprises (OneHotEncoder)": sum(len(c) for c in pre.named_transformers_["cat"].categories_),
    }
pd.DataFrame(learned).round(2)

,pli A (âge moyen 30),pli B (âge moyen 50)
moyenne apprise de l'âge (StandardScaler),29.76,50.26
écart-type appris de l'âge,10.30,11.57
médiane apprise de education_rank (SimpleImputer),2.00,2.00
modalités apprises (OneHotEncoder),41.00,42.00


**Réponse Q3.1.**

| Transformation | Apprend-elle ? | Ce qu'elle apprend (`fit`) |
|---|---|---|
| `clean()` | non | rien : retire des colonnes fixées à l'avance |
| `AddFeatures` | **non** | rien : règles fixes (rangs, seuils, tranches d'âge écrites à la main) |
| `OneHotEncoder` | **oui** | la **liste des modalités** et celles qui sont **rares** (< 20) |
| `SimpleImputer(median)` | **oui** | la **médiane** de chaque colonne numérique |
| `StandardScaler` | **oui** | la **moyenne** et l'**écart-type** de chaque colonne |
| `LogisticRegression` | **oui** | les coefficients |

Le tableau ci-dessus le montre : ajusté sur deux plis différents, le même préprocesseur apprend des paramètres différents (moyenne de l'âge 30 contre 50).

**Pourquoi dans chaque pli, donc dans le pipeline (slide 23) ?** En validation croisée, le pli de validation joue le rôle de données **futures, jamais vues**. Si l'on ajuste le scaler, l'imputer ou l'encodeur sur toutes les lignes *avant* de découper, les moyennes, médianes et listes de modalités contiennent de l'information sur le pli de validation (et sur le test) : c'est une **fuite**, et le score devient optimiste. Placées dans le pipeline, ces étapes sont ré-ajustées par `cross_validate` sur les seules lignes d'entraînement de chaque pli, puis simplement appliquées (`transform`) au pli de validation, exactement comme en production. `AddFeatures` n'apprenant rien, elle ne crée pas de fuite, mais on la met aussi dans le pipeline pour que le modèle final s'applique d'un seul bloc aux données brutes.

### Q3.2 · `handle_unknown="infrequent_if_exist"` et `min_frequency=20`

In [9]:
# Démonstration sur une colonne synthétique : 8 « Dont know », comme dans le vrai jeu
from sklearn.preprocessing import OneHotEncoder

train = pd.DataFrame({"marital_status": ["Married/Living together"] * 120 + ["Single/Never Married"] * 72
                      + ["Dont know"] * 8})
new = pd.DataFrame({"marital_status": ["Single/Never Married", "Dont know", "Jamais vue"]})

variants = {
    "défaut (handle_unknown='error')": OneHotEncoder(sparse_output=False),
    "handle_unknown='ignore'": OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    "min_frequency=20 + infrequent_if_exist": OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist", sparse_output=False),
}
for name, enc in variants.items():
    enc.fit(train)
    try:
        out = pd.DataFrame(enc.transform(new), index=new["marital_status"],
                           columns=enc.get_feature_names_out()).astype(int)
        print(f"\n{name}"); display(out)
    except ValueError as e:
        print(f"\n{name} → ValueError : {str(e)[:90]}…")


défaut (handle_unknown='error') → ValueError : Found unknown categories ['Jamais vue'] in column 0 during transform…

handle_unknown='ignore'


,marital_status_Dont know,marital_status_Married/Living together,marital_status_Single/Never Married
marital_status,,,
Single/Never Married,0,0,1
Dont know,1,0,0
Jamais vue,0,0,0



min_frequency=20 + infrequent_if_exist


,marital_status_Married/Living together,marital_status_Single/Never Married,marital_status_infrequent_sklearn
marital_status,,,
Single/Never Married,0,1,0
Dont know,0,0,1
Jamais vue,0,0,1


**Réponse Q3.2.**
- **`min_frequency=20`** : à l'ajustement, toute modalité vue **moins de 20 fois** dans les lignes d'entraînement est regroupée dans une seule colonne `infrequent_sklearn` au lieu d'avoir la sienne. On évite ainsi des colonnes presque vides, dont le coefficient serait estimé sur une poignée de personnes (variance élevée).
- **`handle_unknown="infrequent_if_exist"`** : à la prédiction, une modalité **jamais vue** à l'entraînement est envoyée dans la colonne « infrequent » si elle existe (sinon elle est codée par des zéros), **sans erreur**. Par défaut (`"error"`), le pipeline plante ; avec `"ignore"`, la ligne est codée en zéros, ce qui revient à la traiter comme la modalité de référence.

**Lien avec Q2.6, cas par cas sur nos données :**
- *Dont know* (`marital_status`, **8 lignes, toutes au Kenya**) : toujours sous le seuil, donc toujours regroupée. En validation croisée **groupée par pays** (étape 4), quand le Kenya est laissé de côté, cette modalité est **inconnue** à l'entraînement : `infrequent_if_exist` évite le plantage.
- *Other/Dont know/RTA* (`education_level`, **35 lignes**) : un pli d'entraînement de la validation croisée à 5 plis n'en contient qu'environ 35 × 0,8 × 0,8 ≈ 22, **juste au-dessus du seuil** : selon le pli, elle aura sa colonne ou sera regroupée. Les deux cas fonctionnent ; on le vérifiera pli par pli à l'étape 4. Dans le **rang ordinal**, elle est traitée à part (NaN → médiane + indicateur `education_unknown`).
- `job_type` : **aucune modalité n'est propre à un seul pays**, donc pas de modalité inconnue en CV groupée par pays. Le problème est ailleurs : les questionnaires diffèrent (pas de *Farming and Fishing* en Tanzanie ni en Ouganda), donc une même modalité ne désigne pas la même population d'un pays à l'autre. Aucun réglage d'encodeur ne corrige ce **décalage de sens** : on le mesurera par la CV groupée (Q4.4) et on le citera dans les limites.

### Q3.3 · `education_level` en one-hot et en rang ordinal

In [10]:
# Ce que chaque encodage donne au modèle
edu = pd.DataFrame({"education_level": [*features.EDU_ORDER, "Other/Dont know/RTA"]})
enc_edu = AddFeatures().fit_transform(make_toy(len(edu), 35).assign(education_level=edu["education_level"].to_numpy()))
onehot = pd.get_dummies(edu["education_level"]).astype(int)
pd.concat({"rang ordinal (1 colonne)": enc_edu[["education_rank", "education_unknown"]],
           "one-hot (6 colonnes)": onehot.set_axis(range(len(edu)))}, axis=1).set_axis(edu["education_level"])

rang ordinal (1 colonne)                    \
                                          education_rank education_unknown   
education_level                                                              
No formal education                                  0.0                 0   
Primary education                                    1.0                 0   
Secondary education                                  2.0                 0   
Vocational/Specialised training                      3.0                 0   
Tertiary education                                   4.0                 0   
Other/Dont know/RTA                                  NaN                 1   

                                one-hot (6 colonnes)                      \
                                 No formal education Other/Dont know/RTA   
education_level                                                            
No formal education                                1                   0   
Primary education                                  0                   0   
Secondary education                                0                   0   
Vocational/Specialised training                    0                   0   
Tertiary education                                 0                   0   
Other/Dont know/RTA                                0                   1   

                                                                       \
                                Primary education Secondary education   
education_level                                                         
No formal education                             0                   0   
Primary education                               1                   0   
Secondary education                             0                   1   
Vocational/Specialised training                 0                   0   
Tertiary education                              0                   0   
Other/Dont know/RTA                             0                   0   

                                                    \
                                Tertiary education   
education_level                                      
No formal education                              0   
Primary education                                0   
Secondary education                              0   
Vocational/Specialised training                  0   
Tertiary education                               1   
Other/Dont know/RTA                              0   

                                                                 
                                Vocational/Specialised training  
education_level                                                  
No formal education                                           0  
Primary education                                             0  
Secondary education                                           0  
Vocational/Specialised training                               1  
Tertiary education                                            0  
Other/Dont know/RTA                                           0

**Réponse Q3.3.**
- **Régression logistique → le rang ordinal aide.** Le modèle est linéaire dans le logit : avec le rang, **un seul coefficient** traduit « chaque niveau en plus multiplie la cote par $e^{\beta}$ ». C'est exactement la forme observée (3,9 % → 8,5 % → 23,3 % → 51–57 %, presque monotone) : peu de paramètres, donc peu de variance. Le one-hot donne un coefficient libre par niveau : plus souple (il capte que la formation professionnelle dépasse légèrement le supérieur), mais il ignore l'ordre. Garder **les deux** permet au modèle d'avoir la tendance (rang) et les écarts à la tendance (one-hot), la régularisation L2 évitant la redondance.
- **Arbre → le rang ordinal aide aussi, d'une autre façon.** Un arbre coupe sur un seuil : avec le rang, une seule coupure `education_rank ≥ 3` sépare « professionnel ou supérieur » du reste. Avec le one-hot, il faut plusieurs coupures successives (une par colonne), ce qui demande plus de profondeur. Pour un arbre, le rang est donc l'encodage le plus efficace ; le one-hot reste utile pour isoler une modalité particulière.
- **Q2.6** : *Other/Dont know/RTA* n'a pas de place dans l'ordre. Le guide la code −1 (sous « aucune éducation »), ce qui invente un ordre ; nous la codons **NaN**, imputée par la médiane apprise dans le pli, avec l'indicateur `education_unknown`. Le one-hot la garde comme modalité à part entière (31 % de ces personnes ont un compte).

### Q3.4 · Le `StandardScaler` : utile pour une forêt ? pour une régression logistique ?

In [11]:
# Expérience sur données synthétiques : deux variables d'échelles très différentes
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

Xs, ys = make_classification(n_samples=2000, n_features=5, n_informative=3, random_state=config.RANDOM_STATE)
Xs = Xs * np.array([1, 10, 100, 1000, 10000])  # échelles de 1 à 10 000
Xs_scaled = StandardScaler().fit_transform(Xs)

rf_raw = RandomForestClassifier(n_estimators=100, random_state=config.RANDOM_STATE).fit(Xs, ys)
rf_std = RandomForestClassifier(n_estimators=100, random_state=config.RANDOM_STATE).fit(Xs_scaled, ys)
lr_raw = LogisticRegression(max_iter=5000).fit(Xs, ys)
lr_std = LogisticRegression(max_iter=5000).fit(Xs_scaled, ys)

pd.DataFrame({
    "forêt aléatoire": {"prédictions identiques brut / standardisé": np.allclose(rf_raw.predict_proba(Xs), rf_std.predict_proba(Xs_scaled)),
                        "itérations de l'optimiseur": "—"},
    "régression logistique": {"prédictions identiques brut / standardisé": np.allclose(lr_raw.predict_proba(Xs), lr_std.predict_proba(Xs_scaled)),
                              "itérations de l'optimiseur": f"{lr_raw.n_iter_[0]} (brut) contre {lr_std.n_iter_[0]} (standardisé)"},
})

,forêt aléatoire,régression logistique
prédictions identiques brut / standardisé,True,False
itérations de l'optimiseur,—,49 (brut) contre 9 (standardisé)


**Réponse Q3.4.**
- **Forêt aléatoire : inutile (mais sans danger).** Un arbre coupe sur des seuils `x ≤ s` ; une transformation croissante de `x` (comme la standardisation) déplace le seuil mais ne change aucune partition. Les prédictions sont **identiques** avec ou sans standardisation (expérience ci-dessus).
- **Régression logistique : utile, pour deux raisons.**
  1. **La régularisation** : par défaut, `LogisticRegression` pénalise $\lambda\sum_j \beta_j^2$ (L2, `C=1`). La taille d'un coefficient dépend de l'unité de sa variable : l'âge (16 à 100) a un petit coefficient, une indicatrice 0/1 un grand. Sans standardisation, la pénalité frappe les variables **inégalement**, pour une raison d'unité et non d'importance ; les prédictions changent (expérience ci-dessus).
  2. **L'optimisation** : l'optimiseur (L-BFGS) converge bien plus vite quand les variables sont à la même échelle.

  La standardisation rend aussi les coefficients **comparables** entre eux, ce qui aide l'interprétation. Ici, les variables numériques (âge, taille du ménage, rang d'éducation) et les indicatrices one-hot (0/1) ont des échelles différentes : on standardise les numériques, dans le pipeline.

### Q3.5 · Variables supplémentaires justifiées par le métier

**Réponse Q3.5.** Nous ajoutons deux variables, en plus de celles du guide :

1. **`phone_education`** = rang d'éducation si la personne a un téléphone, 0 sinon.
   - *Idée métier* : KaziPay vend un **service de paiement mobile** ; il faut à la fois le canal (le téléphone) et la capacité à utiliser des services financiers (l'éducation).
   - *Constat* (étape 2) : sans téléphone, la part des personnes qui ont un compte reste sous 7 % quel que soit le niveau d'éducation ; avec téléphone, elle monte de 5 % à 54 %. Les deux effets **se multiplient** : une régression logistique additive ne peut pas le représenter sans ce terme croisé (un arbre, si).
2. **`lives_alone`** = 1 si le ménage compte une seule personne.
   - *Idée métier* : une personne seule gère elle-même ses revenus et ses paiements.
   - *Constat* : 19,1 % des personnes seules ont un compte, contre 14,1 % en moyenne, alors que l'effet de la taille du ménage est faible ailleurs : un **seuil**, pas une pente.

**Mesure à l'étape 5** : `pipe(..., features="guide")` contre `pipe(..., features="all")` pour la régression logistique, sur les mêmes plis. On ne conclura à un gain que si l'écart de PR-AUC dépasse l'écart-type entre plis.

### Synthèse de l'étape 3

- `clean()` retire `uniqueid` et `year` ; `X` garde 8 colonnes catégorielles et 2 numériques.
- `AddFeatures` ajoute les 5 variables du guide, l'indicateur `education_unknown` (Q2.6) et nos 2 variables (Q3.5), sans rien apprendre.
- Le préprocesseur (one-hot, imputation, standardisation) apprend ses paramètres et vit dans le pipeline, ajusté pli par pli.
- Les pièges de Q2.6 sont traités explicitement et testés (`tests/test_features.py`) : modalités rares regroupées, modalité inconnue sans erreur, *Other/Dont know/RTA* manquante dans le rang ordinal.
- Aucun ajustement sur les vraies données à ce stade : le test sera mis sous scellé au début de l'étape 4.

## Étape 4 · Construire un protocole d'évaluation honnête

*À venir.*